# DSA211: Diagnosing and Remedying Nonlinearity in Bike-Sharing Demand Models
## Notebook 02: Base Data Processing
**Group 2**: Kok Wai Ying, Lee Sin Yang, Lim Wei Lun Marcus, Samuel Teo Yuzhen, Tee Jing Yuan

> **AI Coding Assistant Declaration**:
> In accordance with course guidelines, an AI coding assistant was used strictly for code scaffolding and plotting functions (`*Code: Generated by AI Assistant*`). All modeling decisions, interpretations, and written commentaries are independently authored by group members in the designated sections below (see [`agents.md`](../agents.md)).

---

### Objectives & Workflow Compliance (Referencing `docs/project_cv_workflow.html`)
This notebook reads the training set only (`data/train_clean.csv`). Step 1 (split once) is done in Notebook 00 and the data quality audit in Notebook 01.

It builds the **base processed training dataset** (`data/train_base.csv`) shared by every model in Notebooks 03–06:
1. **Feature roles**: drop leakage, index, and redundant columns.
2. **Dummy encoding**: one-hot encode the integer-coded categorical variables (`hr`, `mnth`, `weekday`, `weathersit`).
3. **Export**: write the base dataset and fix the shared expanding-window CV protocol (Workflow Step 4).

Model-specific transforms (log target, polynomial terms, interactions, scaling) are built in the model notebooks.

## 1. Setup and Environment Initialization
*Code: Generated by AI Assistant*

In [1]:
import os
import sys
import numpy as np
import pandas as pd

# Ensure local source modules can be imported
sys.path.append(os.path.abspath('..'))
from src.data_utils import load_train, TRAIN_PATH, TEST_PATH, BASE_TRAIN_PATH, PROJECT_ROOT
from src.cv_utils import get_cv_splitter, FIRST_VAL_MONTH, LAST_VAL_MONTH

# Load the training set only (written by Notebook 00)
train_df = load_train()
print(f"Training set loaded. Shape: {train_df.shape}")
train_df.head()

Training set loaded. Shape: (15643, 17)


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


## 2. Feature Roles (Workflow Step 3)
*Code: Generated by AI Assistant*

### 2.1 Column Roles
**Excluded Columns**:
- `instant`: Row index.
- `casual`, `registered`: Component targets dropped to prevent target leakage.
- `atemp`: Feeling temperature dropped due to near-perfect collinearity with `temp` ($r = 0.9873$, $\text{VIF} > 300$).
- `season`: Deterministic function of `mnth` ($\text{VIF}_{\text{season}} = 26.0$, $\text{VIF}_{\text{mnth}} = 17.8$); `mnth` is kept.

**Kept, not a predictor**:
- `dteday`: Row identifier for date-based diagnostics (e.g. outlier adjudication).

**Target**: `cnt`

In [2]:
# Define column roles
DROP_COLS = ['instant', 'casual', 'registered', 'atemp', 'season']
ID_COLS = ['dteday']
TARGET = 'cnt'

CATEGORICAL_FEATURES = ['hr', 'mnth', 'weekday', 'weathersit']
BINARY_FEATURES = ['yr', 'holiday', 'workingday']
NUMERIC_FEATURES = ['temp', 'hum', 'windspeed']

# Every column has exactly one role
roles = DROP_COLS + ID_COLS + [TARGET] + CATEGORICAL_FEATURES + BINARY_FEATURES + NUMERIC_FEATURES
assert sorted(roles) == sorted(train_df.columns), set(train_df.columns) ^ set(roles)

print(f"Dropped columns      : {DROP_COLS}")
print(f"ID columns           : {ID_COLS}")
print(f"Target               : {TARGET}")
print(f"Categorical features : {CATEGORICAL_FEATURES}")
print(f"Binary features      : {BINARY_FEATURES}")
print(f"Numeric features     : {NUMERIC_FEATURES}")

Dropped columns      : ['instant', 'casual', 'registered', 'atemp', 'season']
ID columns           : ['dteday']
Target               : cnt
Categorical features : ['hr', 'mnth', 'weekday', 'weathersit']
Binary features      : ['yr', 'holiday', 'workingday']
Numeric features     : ['temp', 'hum', 'windspeed']


### Interpretation & Modeling Decisions: Feature Roles
**Author**: [Insert Name]  
<!-- Document rationale for dropped variables -->

## 3. Dummy Encoding of Categorical Codes (Workflow Step 3)
*Code: Generated by AI Assistant*

`hr`, `mnth`, `weekday`, and `weathersit` are stored as integer codes. Each is one-hot encoded with the first level as the reference category (`drop_first=True`).
- Levels are fixed from the data dictionary (`hr` 0–23, `mnth` 1–12, `weekday` 0–6, `weathersit` 1–4), so the same columns are produced for any subset of rows (CV folds, test set).
- Data-dependent transforms such as `StandardScaler` are not applied here; they are fitted inside model pipelines.

In [3]:
# Fixed category levels (first level = reference category)
CATEGORY_LEVELS = {
    'hr': list(range(24)),
    'mnth': list(range(1, 13)),
    'weekday': list(range(7)),
    'weathersit': [1, 2, 3, 4],
}

for col, levels in CATEGORY_LEVELS.items():
    unexpected = set(train_df[col].unique()) - set(levels)
    assert not unexpected, f"{col} has unexpected levels: {unexpected}"

cat_frame = pd.DataFrame({
    col: pd.Categorical(train_df[col], categories=levels)
    for col, levels in CATEGORY_LEVELS.items()
})
dummies = pd.get_dummies(cat_frame, prefix=CATEGORICAL_FEATURES, drop_first=True, dtype=int)

summary = pd.DataFrame({
    'levels': [len(CATEGORY_LEVELS[c]) for c in CATEGORICAL_FEATURES],
    'reference_level': [CATEGORY_LEVELS[c][0] for c in CATEGORICAL_FEATURES],
    'dummy_columns': [sum(d.startswith(f'{c}_') for d in dummies.columns) for c in CATEGORICAL_FEATURES],
}, index=CATEGORICAL_FEATURES)
print(summary)
print(f"\nTotal dummy columns: {dummies.shape[1]}")

            levels  reference_level  dummy_columns
hr              24                0             23
mnth            12                1             11
weekday          7                0              6
weathersit       4                1              3

Total dummy columns: 43


### Interpretation & Modeling Decisions: Dummy Encoding
**Author**: [Insert Name]  
<!-- Document the choice of dummy encoding and reference categories for hr, mnth, weekday, and weathersit -->

## 4. Assemble & Export Base Training Dataset
*Code: Generated by AI Assistant*

Column order in `data/train_base.csv`: ID (`dteday`), target (`cnt`), dummy columns, binary features, numeric features.

In [4]:
# Assemble base processed training dataset
base_df = pd.concat([
    train_df[ID_COLS + [TARGET]],
    dummies,
    train_df[BINARY_FEATURES + NUMERIC_FEATURES],
], axis=1)

FEATURE_COLS = [c for c in base_df.columns if c not in ID_COLS + [TARGET]]

# Sanity checks
assert len(base_df) == len(train_df)
assert base_df.isnull().sum().sum() == 0
assert all(pd.api.types.is_numeric_dtype(base_df[c]) for c in FEATURE_COLS)
for col in CATEGORICAL_FEATURES:
    group = [c for c in dummies.columns if c.startswith(f'{col}_')]
    assert base_df[group].sum(axis=1).max() <= 1, col

base_df.to_csv(BASE_TRAIN_PATH, index=False)

print(f"Base training dataset shape: {base_df.shape}")
print(f"Predictor columns: {len(FEATURE_COLS)} ({dummies.shape[1]} dummy + {len(BINARY_FEATURES)} binary + {len(NUMERIC_FEATURES)} numeric)")
print(f"Saved to: {BASE_TRAIN_PATH}")
base_df.head()

Base training dataset shape: (15643, 51)
Predictor columns: 49 (43 dummy + 3 binary + 3 numeric)
Saved to: D:\DSA211\data\train_base.csv


,dteday,cnt,hr_1,hr_2,hr_3,hr_4,hr_5,hr_6,hr_7,hr_8,...,weekday_6,weathersit_2,weathersit_3,weathersit_4,yr,holiday,workingday,temp,hum,windspeed
0,2011-01-01,16,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0.24,0.81,0.0
1,2011-01-01,40,1,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0.22,0.80,0.0
2,2011-01-01,32,0,1,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0.22,0.80,0.0
3,2011-01-01,13,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0.24,0.75,0.0
4,2011-01-01,1,0,0,0,1,0,0,0,0,...,1,0,0,0,0,0,0,0.24,0.75,0.0


## 5. Common Cross-Validation Protocol & Evaluation Metric (Workflow Step 4)
*Code: Generated by AI Assistant*

Following `docs/project_cv_workflow.html`, with an expanding-window time-series splitter in place of shuffled `KFold`:
- **Splitter**: `ExpandingWindowSplit` from `src/cv_utils.py`. One validation fold per calendar month from Feb 2012 to Sep 2012 ($k = 8$); each fold trains on all earlier months.
- **Evaluation Metric**: Count-scale MSE and RMSE
- **Standard Error Formula**:
  $$\text{CV Error} = \frac{1}{k}\sum_{i=1}^k \text{MSE}_i, \quad \text{SE} = \frac{\text{SD of fold errors}}{\sqrt{k}}$$
- When evaluating log-transformed target models, back-transform fold predictions ($\exp(\hat{y} + \hat{\sigma}^2/2)$) before computing count-scale metrics.

In [5]:
# Build the shared expanding-window splitter on the base dataset's dates
cv_splitter = get_cv_splitter(base_df['dteday'])
print(f"Shared CV Splitter: ExpandingWindowSplit ({cv_splitter.get_n_splits()} monthly folds, {FIRST_VAL_MONTH} to {LAST_VAL_MONTH})\n")

dates = base_df['dteday']
fold_table = pd.DataFrame([
    {'fold': k + 1,
     'train_start': dates.iloc[tr].min().date(),
     'train_end': dates.iloc[tr].max().date(),
     'train_rows': len(tr),
     'validation_month': dates.iloc[va].min().strftime('%Y-%m'),
     'validation_rows': len(va)}
    for k, (tr, va) in enumerate(cv_splitter.split(base_df))
])
print(fold_table.to_string(index=False))

# Rows after the last validation month are never validated on; they are used only when refitting on all training data
after_last = dates.dt.to_period('M') > pd.Period(LAST_VAL_MONTH, 'M')
print(f"\nRows after {LAST_VAL_MONTH} (refit only): {after_last.sum():,} ({dates[after_last].min().date()} to {dates[after_last].max().date()})")

Shared CV Splitter: ExpandingWindowSplit (8 monthly folds, 2012-02 to 2012-09)

 fold train_start  train_end  train_rows validation_month  validation_rows
    1  2011-01-01 2012-01-31        9386          2012-02              692
    2  2011-01-01 2012-02-29       10078          2012-03              743
    3  2011-01-01 2012-03-31       10821          2012-04              718
    4  2011-01-01 2012-04-30       11539          2012-05              744
    5  2011-01-01 2012-05-31       12283          2012-06              720
    6  2011-01-01 2012-06-30       13003          2012-07              744
    7  2011-01-01 2012-07-31       13747          2012-08              744
    8  2011-01-01 2012-08-31       14491          2012-09              720

Rows after 2012-09 (refit only): 432 (2012-10-01 to 2012-10-18)


### Interpretation & Modeling Decisions: CV Protocol & Metric Selection
**Author**: [Insert Name]  
<!-- Document the expanding-window CV design, count-scale MSE/RMSE metric, and standard error formula -->


## 6. Processed Artifacts Summary & Handoff to Notebook 03
*Code: Generated by AI Assistant*

Summary of shared artifacts for all downstream model notebooks:
- `data/train_clean.csv`: Training dataset from Notebook 00 ($N = 15,643$)
- `data/train_base.csv`: Base processed training dataset from this notebook ($N = 15,643$)
- `data/test_locked.csv`: Locked test holdout from Notebook 00, not read until Notebook 06 ($N = 1,736$)
- `src/cv_utils.py`: Reusable module with `ExpandingWindowSplit` / `get_cv_splitter()`, `evaluate_cv_model()`, and `back_transform_preds()`
- `src/data_utils.py`: Reusable module with `load_train()`, `load_train_base()`, `load_test()`, and `chronological_split()`

In [6]:
# Audit shared artifacts on disk (the test set is checked for existence only, never read)
print("=== Summary of Ready Artifacts ===")
artifacts = [
    (TRAIN_PATH, f"{len(train_df):,} rows"),
    (BASE_TRAIN_PATH, f"{base_df.shape[0]:,} rows x {base_df.shape[1]} columns"),
    (TEST_PATH, "LOCKED - not read in this notebook"),
    (PROJECT_ROOT / 'src' / 'cv_utils.py', "Shared CV & evaluation module"),
    (PROJECT_ROOT / 'src' / 'data_utils.py', "Shared data loading module")
]

for p, desc in artifacts:
    exists = os.path.exists(p)
    size_kb = os.path.getsize(p) / 1024 if exists else 0
    status = "EXISTS" if exists else "MISSING"
    print(f"[{status}] {p.relative_to(PROJECT_ROOT)} ({size_kb:.1f} KB) -> {desc}")

=== Summary of Ready Artifacts ===
[EXISTS] data\train_clean.csv (1018.8 KB) -> 15,643 rows
[EXISTS] data\train_base.csv (1889.9 KB) -> 15,643 rows x 51 columns
[EXISTS] data\test_locked.csv (115.8 KB) -> LOCKED - not read in this notebook
[EXISTS] src\cv_utils.py (4.8 KB) -> Shared CV & evaluation module
[EXISTS] src\data_utils.py (2.1 KB) -> Shared data loading module


### Summary & Handoff to Notebook 03 (Baseline Model)
**Author**: [Insert Name]  
<!-- Concluding remarks on data processing readiness and bridge to Notebook 03: Baseline Model & Failure Diagnosis -->
